# 02 · Etiquetado físico: de β a visibilidad en metros (Koschmieder)

**Camanchaca-Predict — G5 · Proyecto Aplicado 2026-2**

**El diferenciador del proyecto:** ningún dataset de niebla entrega visibilidad
en metros; nosotros la derivamos físicamente. Este notebook:

1. Repasa la ley de Koschmieder y el modelo atmosférico de dispersión (ASM).
2. Convierte el β de cada imagen del manifiesto en `V = 3.912/β` metros.
3. Asigna bandas de seguridad vial justificadas con la distancia de detención.
4. **Valida** la consistencia de las etiquetas (proxy de canal oscuro).
5. Genera `labels.csv` — la tabla que alimenta los modelos.

Requiere: `data/processed/reside_manifest.csv` (notebook 01).

In [ ]:
# Setup estándar del proyecto
import math, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/camanchaca")
except ImportError:
    ROOT = Path("local_workspace")

DATA_DIR = ROOT / "data"
MANIFEST_PATH = DATA_DIR / "processed" / "reside_manifest.csv"
LABELS_PATH = DATA_DIR / "processed" / "labels.csv"

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

assert MANIFEST_PATH.exists(), "Ejecuta primero el notebook 01 (falta reside_manifest.csv)"
manifest = pd.read_csv(MANIFEST_PATH)
print(f"Manifiesto: {len(manifest)} imágenes · {manifest['scene'].nunique()} escenas")

## 1. Marco físico

### Ley de Koschmieder
Un objeto oscuro a distancia `d` contra el cielo tiene contraste aparente
`C(d) = exp(-β·d)`. La **visibilidad** es la distancia donde el contraste cae
al umbral ε (usamos **ε = 0.02**, conservador para seguridad vial):

$$V = -\ln(\varepsilon)/\beta \;\approx\; 3.912/\beta \quad [\text{metros}]$$

### Modelo atmosférico de dispersión (ASM)
RESIDE sintetiza niebla con `I(x) = J(x)·t(x) + A·(1−t(x))` y
`t(x) = exp(−β·d(x))`, registrando β en el nombre de archivo. Ese β
[1/m] es nuestra ancla física.

**Sensibilidad del umbral:** con ε = 0.05 (MOR de aviación), V = 3.0/β
(~23 % menor). Lo reportamos como análisis de sensibilidad, no cambiamos el
criterio principal.

In [ ]:
# Funciones del criterio de etiquetado (idénticas a src/camanchaca/labeling/)
DEFAULT_EPSILON = 0.02
BAND_EDGES = (50.0, 100.0, 200.0)
BAND_NAMES = ("critico", "alto_riesgo", "precaucion", "aceptable")
BAND_LABELS_ES = ("Crítico (<50 m)", "Alto riesgo (50–100 m)",
                  "Precaución (100–200 m)", "Aceptable (≥200 m)")

def visibility_from_beta(beta, epsilon=DEFAULT_EPSILON):
    beta = np.asarray(beta, dtype=float)
    return -np.log(epsilon) / beta

def band_from_visibility(v):
    v = np.asarray(v, dtype=float)
    return np.array(BAND_NAMES)[np.digitize(v, BAND_EDGES)]

def stopping_sight_distance(v_kmh, t_reaction=2.5, mu=0.4, g=9.81):
    v = np.asarray(v_kmh, dtype=float) / 3.6
    return v * t_reaction + v ** 2 / (2 * mu * g)

# Ejemplos numéricos de la conversión
ejemplos = pd.DataFrame({
    "beta_1_sobre_m": [0.078, 0.039, 0.02, 0.01, 0.005],
})
ejemplos["V_metros"] = visibility_from_beta(ejemplos["beta_1_sobre_m"]).round(1)
ejemplos["banda"] = [str(b) for b in band_from_visibility(ejemplos["V_metros"])]
ejemplos

In [ ]:
# Justificación de las bandas: distancia de detención (SSD) en piso mojado
ssd = pd.DataFrame({
    "velocidad_kmh": [40, 60, 80, 100, 120],
    "SSD_m": stopping_sight_distance([40, 60, 80, 100, 120]).round(1),
})
display(ssd)
print("Lectura: con V < 50 m ni frenando a 60 km/h se detiene a tiempo;")
print("100-200 m da margen hasta ~100 km/h; >=200 m incluso a 120 km/h.")

## 2. Construcción de labels.csv

Columnas generadas:

| Columna | Significado |
|---|---|
| `visibility_m` | etiqueta física V = 3.912/β |
| `log10_v` | target de regresión (estabiliza el rango ~10–2000 m) |
| `band`, `band_idx` | etiqueta de clasificación (0..3) |

In [ ]:
# Conversión β -> V -> banda para todo el manifiesto
labels = manifest.copy()
labels["visibility_m"] = visibility_from_beta(labels["beta"].values)
labels["log10_v"] = np.log10(labels["visibility_m"])
labels["band"] = [str(b) for b in band_from_visibility(labels["visibility_m"].values)]
labels["band_idx"] = np.digitize(labels["visibility_m"].values, BAND_EDGES)
labels["V_metros_eps05"] = -np.log(0.05) / labels["beta"]   # sensibilidad ε=0.05

display(labels[["image", "scene", "beta", "visibility_m", "log10_v", "band"]].head())
print(f"V: min={labels['visibility_m'].min():.1f} m · "
      f"mediana={labels['visibility_m'].median():.1f} m · "
      f"max={labels['visibility_m'].max():.1f} m")

In [ ]:
# Distribución de la visibilidad y balance de bandas
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].hist(labels["visibility_m"], bins=60, color="#3681a6", edgecolor="white")
for e in BAND_EDGES:
    axes[0].axvline(e, color="crimson", ls="--", lw=1)
axes[0].set_xlabel("Visibilidad [m]"); axes[0].set_ylabel("imágenes")
axes[0].set_title("Distribución de V (cortes de banda en rojo)")

counts = labels["band"].value_counts().reindex(BAND_NAMES, fill_value=0)
axes[1].bar(BAND_LABELS_ES, counts.values, color=["#b2182b", "#ef8a62", "#67a9cf", "#2166ac"])
axes[1].set_title("Imágenes por banda de seguridad")
axes[1].tick_params(axis="x", rotation=20)
for i, c in enumerate(counts.values):
    axes[1].text(i, c, f"{c}\n({100*c/len(labels):.0f}%)", ha="center", va="bottom", fontsize=9)
plt.show()

print("⚠️ Si 'critico' queda < 5% -> activar mitigación R3 (class weights / re-síntesis).")

## 3. Validación de consistencia de las etiquetas

Las etiquetas dependen del β del nombre de archivo. **Validación perceptual:**
el canal oscuro (He et al. 2011) de una imagen con niebla densa es más
brillante (se acerca a la luz atmosférica). Si nuestro β es correcto, la
correlación de Spearman entre el proxy y β debe ser **alta y positiva**.

In [ ]:
# Proxy de niebla (canal oscuro) vs beta: correlación de Spearman
import cv2
from scipy.stats import spearmanr

def haze_density_proxy(img_bgr, patch=15):
    kernel = np.ones((patch, patch), np.uint8)
    dc = cv2.erode(img_bgr.min(axis=2), kernel)
    return float(dc.mean()) / 255.0

muestra = labels.sample(min(80, len(labels)), random_state=SEED)
proxies, betas = [], []
for _, row in muestra.iterrows():
    img = cv2.imread(row["path"])
    if img is None:
        continue
    proxies.append(haze_density_proxy(img))
    betas.append(row["beta"])

rho, pval = spearmanr(betas, proxies)
print(f"Spearman ρ(beta, proxy_canal_oscuro) = {rho:.3f}  (p = {pval:.2e}, n = {len(betas)})")

fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
ax.scatter(betas, proxies, s=18, alpha=0.7, color="#3681a6")
ax.set_xlabel("β del filename [1/m]"); ax.set_ylabel("proxy canal oscuro [0-1]")
ax.set_title(f"Consistencia perceptual (ρ = {rho:.2f})")
plt.show()

if rho > 0.7:
    print("✅ Etiquetas consistentes con la percepción de niebla.")
elif rho > 0.4:
    print("🟡 Correlación moderada: revisar subconjuntos / outliers (riesgo R2).")
else:
    print("🔴 Correlación débil: activar mitigación R2 (re-etiquetar por re-síntesis).")

**Validación física adicional (opcional, si se consigue ITS + depth maps):**
renderizar de nuevo la niebla con `t = exp(−β·d)` usando profundidad conocida
y comparar contra la imagen entregada confirma unidades y fórmula del
generador. Detalle en `docs/labeling_criteria.md` §5.

In [ ]:
# Guardamos labels.csv (Drive + copia local al repo si está clonado)
labels.to_csv(LABELS_PATH, index=False)
print(f"labels.csv guardado: {LABELS_PATH} ({len(labels)} filas)")

REPO_LOCAL = Path("/content/camanchaca-predict")
if REPO_LOCAL.exists():
    dest = REPO_LOCAL / "data" / "processed" / "labels.csv"
    dest.parent.mkdir(parents=True, exist_ok=True)
    labels.to_csv(dest, index=False)
    print(f"Copia local en el repo: {dest}")

## Resumen del notebook

- ✅ Etiqueta física por imagen: `V = 3.912/β` metros (+ sensibilidad ε=0.05).
- ✅ Bandas de seguridad con justificación SSD (50/100/200 m).
- ✅ Validación de consistencia perceptual (ρ de Spearman).
- ⚠️ Observar el balance de bandas → riesgo R3 si `critico` es minoritaria.

**Siguiente:** notebook 03 · eda_splits (EDA + split 70/15/15 agrupado por
escena, anti-fuga).